# RVL-CDIP OpenRouter multi-model eval (PoC)

Follow-on to [`rvl_cdip_recreation_sampling.ipynb`](rvl_cdip_recreation_sampling.ipynb).

Loads the recreation experiment set (`recreation_samples.jsonl`, typically **1040** rows =
65 × 16 classes), materializes page images (selective tar extract when the archive is present),
optionally OCRs them, then queries **OpenRouter** models to:

1. **Classify** into the 16 RVL-CDIP labels (gold labels available)
2. **Extract / annotate** structured insurance-style fields (proxy metrics — no gold fields)

**Modality:** vision-first (page image → vision models); OCR text → text models.

Related: [RVL-CDIP SQL Index](../rvl_cdip_sql.md) · CLI `python -m src.rvl_cdip --help`

## 0. Prerequisites

```bash
python scripts/setup_env.py          # OPENROUTER_API_KEY in .env
python -m src.rvl_cdip build         # labels + SQLite (if not already)
pip install -e ".[ocr]"             # only if using the text/OCR path

# Image archive (~38 GB) — preview then dual opt-in (writes ONLY under .venv/rvl_cdip):
python -m src.rvl_cdip download-images --preflight
python -m src.rvl_cdip download-images \
  --i-understand-large-download --confirm-writes-under-venv
```

Defaults below keep **`DRY_RUN=True`** so this notebook is safe for docs/CI.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    for cand in [REPO_ROOT.parent, *REPO_ROOT.parents]:
        if (cand / "pyproject.toml").exists():
            REPO_ROOT = cand
            break

import sys

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.rvl_cdip.download import (
    download_images,
    format_image_download_preflight,
    image_download_preflight,
)
from src.rvl_cdip.openrouter_eval import (
    DEFAULT_TEXT_MODELS,
    DEFAULT_VISION_MODELS,
    load_samples,
    run_eval,
)
from src.rvl_cdip.paths import LABEL_NAMES, archive_path
from src.rvl_cdip.sample_images import materialize_samples, summarize_materialize
from src.rvl_cdip.store import RvlCdipStore
from src.utils.config import Config

SAMPLES_PATH = REPO_ROOT / "data/notebook_demo/rvl_cdip_recreation/recreation_samples.jsonl"
OUT_DIR = REPO_ROOT / "data/notebook_demo/rvl_cdip_openrouter_eval"

# --- knobs -------------------------------------------------------------------
DRY_RUN = True          # False → live OpenRouter calls (needs OPENROUTER_API_KEY)
MAX_PER_CLASS = 65      # full PoC = 65 → 1040 docs; use 2–5 for cheap smoke
RUN_OCR = True          # needed for text modality
MODALITIES = ("vision", "text")
VISION_MODELS = list(DEFAULT_VISION_MODELS)
TEXT_MODELS = list(DEFAULT_TEXT_MODELS)

cfg = Config.load()
has_key = bool(cfg.openrouter_api_key)
display(
    Markdown(
        f"Repo `{REPO_ROOT}`  \n"
        f"`OPENROUTER_API_KEY` set: **{has_key}**  ·  `DRY_RUN={DRY_RUN}`  ·  "
        f"`MAX_PER_CLASS={MAX_PER_CLASS}`  ·  archive present: **{archive_path().is_file()}**"
    )
)

## 0b. Optional: download image archive into `.venv/rvl_cdip` only

Hub cache + `rvl-cdip.tar.gz` write **only** under `.venv/rvl_cdip/` (never `data/` or
`~/.cache`). Set **both** flags below to `True` after reviewing the preflight block,
then run the cell. Prefer CLI for long downloads:

```bash
python -m src.rvl_cdip download-images --preflight
python -m src.rvl_cdip download-images \
  --i-understand-large-download --confirm-writes-under-venv
# interactive phrase prompt instead of the second flag:
# python -m src.rvl_cdip download-images \
#   --i-understand-large-download --interactive-confirm
```

In [ ]:
I_UNDERSTAND_LARGE_DOWNLOAD = False   # ~38 GB archive
CONFIRM_WRITES_UNDER_VENV = False     # must acknowledge .venv/rvl_cdip-only writes
FORCE_REDOWNLOAD = False

preflight = image_download_preflight(force=FORCE_REDOWNLOAD)
print(format_image_download_preflight(preflight))
display(preflight)

if I_UNDERSTAND_LARGE_DOWNLOAD and CONFIRM_WRITES_UNDER_VENV:
    if not preflight["enough_free_space"] and not preflight["archive_already_present"]:
        raise RuntimeError(
            f"Need ≥ {preflight['min_free_gb']} GB free; have {preflight['free_gb']} GB."
        )
    result = download_images(
        force=FORCE_REDOWNLOAD,
        i_understand_large_download=True,
        confirm_writes_under_venv=True,
    )
    updated = RvlCdipStore().refresh_image_paths()
    display(
        {
            "local_path": str(result.local_path),
            "bytes": result.bytes,
            "skipped": result.skipped,
            "image_abspath_updated": updated,
            "writes_only_under": result.detail.get("writes_only_under"),
        }
    )
else:
    display(
        Markdown(
            "> **Download not started.** Set `I_UNDERSTAND_LARGE_DOWNLOAD = True` and "
            "`CONFIRM_WRITES_UNDER_VENV = True` after reviewing the preflight "
            "(writes only under `.venv/rvl_cdip`)."
        )
    )

## 1. Load recreation samples

In [ ]:
samples = load_samples(SAMPLES_PATH, max_per_class=MAX_PER_CLASS)
df = pd.DataFrame(samples)
display(Markdown(f"Loaded **{len(samples)}** samples from `{SAMPLES_PATH.relative_to(REPO_ROOT)}`"))
display(df.groupby(["label_id", "label"], as_index=False).size().head(20))
df.head(3)

## 2. Materialize images (+ optional OCR)

Resolution order: existing `image_abspath` → `.venv/rvl_cdip/source/images/{relpath}` →
extract that member from `rvl-cdip.tar.gz` when the archive is present.

In [ ]:
mats = materialize_samples(samples, run_ocr=RUN_OCR and ("text" in MODALITIES))
mat_summary = summarize_materialize(mats)
display(Markdown("### Materialize summary"))
display(mat_summary)
mat_df = pd.DataFrame([m.to_dict() for m in mats])
display(mat_df.head(5))
if mat_summary["n_with_image"] == 0 and "vision" in MODALITIES and not DRY_RUN:
    display(
        Markdown(
            "> **Vision path blocked:** no images on disk and archive missing. "
            "Use section **0b** or\n"
            "`python -m src.rvl_cdip download-images --i-understand-large-download "
            "--confirm-writes-under-venv`, then re-run materialize "
            "(selective extract of the sample paths into `.venv/rvl_cdip/source/images/`)."
        )
    )

## 3. Run OpenRouter eval (classify + extract/annotate)

Two calls per document × model × modality. Predictions append to
`predictions.jsonl` with resume/cache on `(document_id, model_id, task, modality)`.

In [ ]:
if not DRY_RUN and not has_key:
    raise RuntimeError("Set OPENROUTER_API_KEY (or keep DRY_RUN=True)")

manifest = run_eval(
    samples,
    materialize=mats,
    vision_models=VISION_MODELS,
    text_models=TEXT_MODELS,
    modalities=MODALITIES,
    out_dir=OUT_DIR,
    dry_run=DRY_RUN,
    cfg=cfg,
    resume=True,
)
display(Markdown(f"Wrote artifacts under `{OUT_DIR.relative_to(REPO_ROOT)}`"))
display(manifest)

## 4. Scoreboards

- **Classification:** accuracy + macro recall vs gold `label`
- **Extraction:** valid-JSON rate, field fill-rates, pairwise agreement (no gold fields)

In [ ]:
cls_path = OUT_DIR / "summary_classification.json"
ext_path = OUT_DIR / "summary_extraction.json"
cls = json.loads(cls_path.read_text()) if cls_path.exists() else {}
ext = json.loads(ext_path.read_text()) if ext_path.exists() else {}

cls_df = pd.DataFrame(cls.get("per_model") or [])
ext_df = pd.DataFrame(ext.get("per_model") or [])
display(Markdown("### Classification"))
display(cls_df)
display(Markdown("### Extraction proxies"))
display(ext_df)
if ext.get("pairwise_agreement"):
    display(Markdown("### Pairwise agreement"))
    display(pd.DataFrame(ext["pairwise_agreement"].get("pairs") or []))

## 5. Peek at predictions

In [ ]:
pred_path = OUT_DIR / "predictions.jsonl"
preds = [
    json.loads(line)
    for line in pred_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
pred_df = pd.DataFrame(preds)
display(Markdown(f"**{len(pred_df)}** prediction rows"))
cols = [
    c
    for c in [
        "document_id",
        "label",
        "task",
        "modality",
        "model_id",
        "prediction",
        "error",
        "dry_run",
        "cache_hit",
    ]
    if c in pred_df.columns
]
display(pred_df[cols].head(12))

## Next steps

- Download archive via section **0b** / CLI (`--confirm-writes-under-venv`), then materialize
- Set `DRY_RUN=False` for a live vision run
- Lower `MAX_PER_CLASS` (e.g. `2`) for a cheap smoke before the full 1040
- Compare `summary_classification.csv` across model slugs; review extraction JSONL manually
- Guide: [RVL-CDIP SQL Index](../rvl_cdip_sql.md)